# 03: 标准化 + 高可变基因选择（批次感知）

在 02 合并后的 counts 矩阵上执行标准化管线，为后续 04 降维与 05 聚类准备数据。

**标准化**消除文库大小差异——不同细胞的测序深度天然不同，不做标准化会导致
高深度细胞主导后续所有分析。**高可变基因（HVG）选择**决定下游分析使用哪些基因——
只保留携带细胞类型差异信息的高可变基因，大幅降噪并节省内存。
**batch-aware HVG** 是整合分析的关键 trick：在每个批次（source_dataset）内独立选 HVG，
取跨批次共识——防止某个数据集的技术噪声基因霸占 HVG 列表。

**本 notebook 新增能力**（v2 科研工作台升级）：
- 标准化双模：经典 `normalize_total + log1p` 或 Pearson residuals（Lause 2021）
- HVG Scalar-or-Sweep 双模：写单值直接跑，写列表自动对比不同参数组合的 HVG 重叠度
- 批次感知 HVG：`batch_key` 指定按哪个列独立选 HVG
- HVG 排除列表：自动排除线粒体/核糖体/血红蛋白基因，避免技术性基因驱动聚类
- 可选回归混杂变量（`regress_out`）与缩放（`scale`），默认关闭

**本 notebook 产出**：
- `adata.layers['counts']` — 原始 counts，为下游 scVI / scANVI / DESeq2 保留
- `adata.X` — 标准化后的表达矩阵（float32）
- `adata.var['highly_variable']` — HVG 布尔掩码
- `adata.var['hvg_{flavor}_{n}']` — 各参数组合的 HVG 掩码（sweep 模式）
- `adata.uns['normalize_v1']` — 标准化参数记录
- 03 checkpoint `.h5ad` 文件，供 04 嵌入使用

In [ ]:
# === PARAMS ===

UPSTREAM_PATH = "results/02_merged_v1.h5ad"
OUTPUT_PATH   = "results/03_normalized_v1.h5ad"

# --- 标准化方法 ---
NORMALIZATION_METHOD = "standard"   # "standard" | "pearson_residuals"
                                    # standard = normalize_total + log1p（经典，适合大部分场景）
                                    # pearson_residuals = Lause 2021，更好的方差稳定化，跳过 log1p
TARGET_SUM = 1e4                    # normalize_total 的 target（仅 standard 模式生效）

# --- HVG 选择（支持 Scalar-or-Sweep 双模）---
# 写单值直接跑，写列表自动 sweep + 输出 HVG 集合重叠度对比
N_TOP_GENES = 3000                  # 单值 | 列表如 [1500, 2000, 3000, 4000]
                                    # 整合分析 + 组织复杂度高（胃全层 15-20 种细胞类型）→ 3000 保证稀有群体覆盖
HVG_FLAVOR  = "seurat"             # 单值 | 列表如 ["seurat", "seurat_v3"]

# --- 批次感知 HVG（多数据集整合的关键 trick）---
# 防止某个数据集的技术噪声基因霸占 HVG 列表
BATCH_AWARE_HVG = True
HVG_BATCH_KEY   = "source_dataset"  # 按哪个列做 batch-aware

# --- HVG 排除列表 ---
# 这些基因类别不应驱动聚类（它们有信息量但反映的是技术/通用状态而非细胞身份）
EXCLUDE_MT_FROM_HVG   = True        # 线粒体基因（MT-*）
EXCLUDE_RIBO_FROM_HVG = True        # 核糖体基因（RPS*/RPL*）
EXCLUDE_HB_FROM_HVG   = True        # 血红蛋白基因（HBA*/HBB*）
CUSTOM_EXCLUDE_PATTERNS = []        # 额外排除 pattern，如 ["^IG[HKL]", "^TR[ABGD]"]（免疫球蛋白/TCR）

# --- HVG 等量子采样 ---
# None=不采样（直接全量选 HVG）| int=每 batch 先采样 N 个细胞再选 HVG
# 解决细胞数不平衡时大数据集主导 HVG 排名的问题
# 建议值：min(各 batch 细胞数) 或 500-2000
HVG_SUBSAMPLE_PER_BATCH = None

EXCLUDE_CELL_CYCLE_FROM_HVG = False  # True=将 Tirosh 2015 S+G2M 基因加入排除列表（增殖信号主导 PCA 时启用）

FORCED_INCLUDE_GENES = []           # PI 按需填入关键基因，如 ["CDX2", "TFF3", "GKN1", "MUC2", "OLFM4", "LGR5"]
                                    # 强制纳入 HVG——保证研究者关心的关键生物学轴（如炎-癌转化 marker）一定进入下游 PCA 空间

# --- 回归混杂变量（可选，慎用）---
# 空列表 = 不回归。回归会 densify 矩阵、大幅增加内存和时间。
# 通常不推荐在此阶段回归——更好的做法是在 04 嵌入中通过 batch_key 处理。
# 仅在 PI 确认某个混杂因素严重干扰 HVG/PCA 时启用。
REGRESS_OUT = []                    # 可选：["pct_counts_mt", "total_counts", "S_score", "G2M_score"]

# --- 缩放（可选）---
# Harmony/scVI 不需要 scale；PCA-only 管线可能需要。
# 注意：scale 会 densify 矩阵！
SCALE = False
MAX_SCALE_VALUE = 10

OUTPUT_VERSION = 1
RANDOM_SEED    = 42

In [ ]:
# === Setup：sys.path + 导入依赖 ===
# sys.path 必须在 scanpy 导入之前设置，否则找不到 scrna_integration 模块。
import sys, os

_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# --- 所有 import 集中在这里 ---
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import gc
import re
import itertools

np.random.seed(RANDOM_SEED)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

# 环境自检（每次运行自动检测平台 + conda 环境 + 关键包，见 ADR-0012）
try:
    from scrna_integration.platform import env_check
    env_check()
except Exception as _e:
    print(f"环境自检跳过（env_check 不可用: {_e}）")


In [ ]:
# === 加载上游 ===
print("Loading upstream:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

# 宽容整数检查：验证值为整数（不仅看 dtype）
# 上游 02 写出的是 float32（值为整数），dtype.kind 判断会误报。
# 用 allclose 校验值而非 dtype——容忍 float32 表示的整数。
_sample_x = adata.X[:500, :500]
if sp.issparse(_sample_x):
    _sample_x = _sample_x.toarray()
_is_integer_values = np.allclose(_sample_x, np.round(_sample_x), atol=0.01)

if _is_integer_values:
    print(f"X 含整数值 counts（dtype={adata.X.dtype}），符合上游合约")
else:
    print(f"⚠️ X 含非整数值（dtype={adata.X.dtype}）——上游可能已做过标准化变换；确认是否需要调整上游管线")


## 保留原始 counts

标准化操作会改变 `adata.X` 中的原始计数数据。但下游方法（scVI、scANVI、
DESeq2、pseudobulk）需要原始整数计数来精准建模——因此先把 counts 拷贝到
`adata.layers['counts']` 妥善保存。

In [ ]:
# 将原始 counts 保留到 layers['counts']。
# 下游方法（scVI / scANVI / DESeq2 / pseudobulk）需要原始计数数据，
# 而非标准化后的数据。
print("Copying raw counts to adata.layers['counts']...")
adata.layers["counts"] = adata.X.copy()
print(f"layers keys: {list(adata.layers.keys())}")
print(f"counts layer dtype: {adata.layers['counts'].dtype}")

## 标准化

两种方法路线可选（通过 `NORMALIZATION_METHOD` 切换）：

- **`standard`**（默认）：`normalize_total` 将每个细胞缩放至相同总 UMI，消除测序深度差异；
  接着 `log1p` 做方差稳定化，将右偏分布拉近正态，使高表达基因不过度主导 PCA。
  这是单细胞领域的经典路线，适用绝大多数场景。
- **`pearson_residuals`**（Lause et al. 2021, Genome Biology）：基于负二项模型的残差变换，
  一步完成方差稳定化，不需要显式 log1p。优点是对 dropout 更鲁棒；
  缺点是不保留 log-normalized 空间的可解释性，且函数在 `sc.experimental` 中。

In [ ]:
# === 标准化分支 ===
if NORMALIZATION_METHOD == "standard":
    # normalize_total：每个细胞缩放至相同总 UMI，消除测序深度差异
    sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
    # log1p：log(1+x) 方差稳定化，使高表达基因不过度主导 PCA
    sc.pp.log1p(adata)
    print(f"✓ 标准化完成：normalize_total(target_sum={TARGET_SUM}) + log1p")
    print(f"  标准化后 X mean={adata.X.mean():.4f}, max={adata.X.max():.4f}")
elif NORMALIZATION_METHOD == "pearson_residuals":
    # Pearson residuals（Lause et al. 2021, Genome Biology）
    # 一步完成方差稳定化，不需要单独 log1p
    # 优点：对 dropout 更鲁棒；
    # 注意：sc.experimental API 可能在 scanpy 未来版本中变动
    sc.experimental.pp.normalize_pearson_residuals(adata)
    print("✓ 标准化完成：Pearson residuals (Lause 2021)")
    print(f"  标准化后 X mean={adata.X.mean():.4f}, std={adata.X.std():.4f}")
else:
    raise ValueError(f"不支持的 NORMALIZATION_METHOD: {NORMALIZATION_METHOD}，请使用 'standard' 或 'pearson_residuals'")

# 转为 float32——内存减半，单细胞数据有效精度无实质影响
adata.X = adata.X.astype(np.float32)
print(f"X dtype 已转为: {adata.X.dtype}")

## 标准化效果可视化

**看什么**：左图展示标准化前各细胞的总 UMI 计数分布（来自 counts layer）——
不同细胞测序深度差异悬殊。右图展示标准化后的表达值分布——
standard 模式下所有细胞缩放到统一尺度，pearson_residuals 模式下残差围绕 0 对称分布。

In [ ]:
# === 标准化前后对比图 ===
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：标准化前——从 counts layer 看每个细胞的总 UMI 分布
counts_before = np.array(adata.layers["counts"].sum(axis=1)).flatten()
axes[0].hist(counts_before, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(np.median(counts_before), color="red", linestyle="--",
                label=f'中位={np.median(counts_before):.0f}')
axes[0].set_xlabel("总 UMI 计数")
axes[0].set_ylabel("细胞数")
axes[0].set_title("标准化前\n各细胞总 UMI 计数差异大")
axes[0].legend(fontsize=9)

# 右图：标准化后——表达值分布
if sp.issparse(adata.X):
    sample_vals = adata.X.data[:200000] if len(adata.X.data) > 200000 else adata.X.data
else:
    # Dense 矩阵（Pearson residuals 可能产出 dense）——随机采样避免 OOM
    # .flatten() 对 100k×30k=3B float32≈12GB 直接 OOM；随机采 20 万点足够画直方图
    _flat_size = adata.X.shape[0] * adata.X.shape[1]
    if _flat_size > 200000:
        _sample_idx = np.random.choice(_flat_size, size=200000, replace=False)
        sample_vals = adata.X.ravel()[_sample_idx]
    else:
        sample_vals = adata.X.ravel()
axes[1].hist(sample_vals, bins=100, color="coral", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("标准化后表达值")
axes[1].set_ylabel("频数")
axes[1].set_title(f"标准化后（{NORMALIZATION_METHOD}）\n表达值分布")
axes[1].axvline(0, color="gray", linestyle=":", alpha=0.5)

plt.suptitle("标准化效果", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/03_normalize_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

# --- standard 模式额外诊断：top-3 基因标准化前后分布对比 ---
# 这是经典的 log1p 效果展示——看 log1p 如何将右偏分布拉近正态
if NORMALIZATION_METHOD == "standard":
    # 基于原始 counts 均值排名前 3 的基因
    top_idx = np.argsort(
        np.array(adata.layers["counts"].mean(axis=0)).flatten()
    )[-3:]
    top_genes = adata.var_names[top_idx].tolist()
    print(f"\ntop-3 基因表达分布对比: {top_genes}")

    # 重新计算 pre-log1p 标准化值（normalize_total 后、log1p 前）
    raw_top = adata.layers["counts"][:, top_idx].toarray()
    lib_size = np.array(adata.layers["counts"].sum(axis=1)).flatten()
    norm_expr = raw_top / lib_size[:, None] * TARGET_SUM

    # post-log1p 直接读取当前 adata.X
    log_expr = adata.X[:, top_idx].toarray()

    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))
    for i, gene in enumerate(top_genes):
        axes2[0].hist(norm_expr[:, i], bins=50, alpha=0.5, label=gene, density=True)
        axes2[1].hist(log_expr[:, i], bins=50, alpha=0.5, label=gene, density=True)
    axes2[0].set_xlabel("标准化后表达量（未 log）")
    axes2[0].set_ylabel("密度")
    axes2[0].set_title("Log1p 前：高度右偏\n少数细胞极高值主导")
    axes2[0].legend(fontsize=8)
    axes2[1].set_xlabel("log1p 表达量")
    axes2[1].set_ylabel("密度")
    axes2[1].set_title("Log1p 后：接近正态\n适合 PCA 等线性方法")
    axes2[1].legend(fontsize=8)
    plt.suptitle("Log1p 方差稳定化效果", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("results/figures/03_log1p_before_after.png", dpi=150, bbox_inches="tight")
    plt.show()

## 高可变基因选择（Scalar-or-Sweep 双模）

大多数基因在所有细胞中表达量相近（管家基因），不携带区分细胞类型的信息。
高可变基因（HVG）是那些在不同细胞间表达差异最大的基因——
它们携带了细胞类型、状态差异的核心信号。

**双模设计**：
- **单值模式**：`N_TOP_GENES=2000`，直接选一组 HVG，干净利落
- **Sweep 模式**：`N_TOP_GENES=[1500, 2000, 3000]`，自动遍历所有参数组合，
  输出 Jaccard 重叠度热力图，帮助 PI 判断参数敏感性

**批次感知 HVG**：当 `BATCH_AWARE_HVG=True` 时，scanpy 在每个 batch 内独立选 HVG，
取跨批次并集并按出现频率排名。这防止某个数据集的技术噪声基因因全局高变而被选中——
是多数据集整合的关键 trick。

In [ ]:
# === HVG 选择：Scalar-or-Sweep 双模 ===
# 写单值直接跑，写列表自动对比不同参数组合的 HVG 集合

_n_genes_values = N_TOP_GENES if isinstance(N_TOP_GENES, list) else [N_TOP_GENES]
_flavor_values = HVG_FLAVOR if isinstance(HVG_FLAVOR, list) else [HVG_FLAVOR]

# --- Pearson residuals 兼容性 ---
# seurat flavor 假设 count 数据的 mean-variance 关系，
# 对 Pearson 残差（均值~0、可为负值）不兼容。自动切换到 seurat_v3（基于方差排名）
if NORMALIZATION_METHOD == "pearson_residuals":
    _flavor_values_safe = []
    for f in _flavor_values:
        if f == "seurat":
            print("⚠️ Pearson residuals + seurat flavor 不兼容（seurat 假设 count 数据），自动切换到 seurat_v3")
            _flavor_values_safe.append("seurat_v3")
        else:
            _flavor_values_safe.append(f)
    _flavor_values = _flavor_values_safe

hvg_results = {}  # 存储各组合的结果供对比

# HVG 等量子采样：平衡各 batch 对 HVG 选择的影响
# 当数据集细胞数严重不平衡时，大 batch 会主导 HVG 排名——采样子集后再选可缓解
if HVG_SUBSAMPLE_PER_BATCH is not None and BATCH_AWARE_HVG:
    np.random.seed(RANDOM_SEED)
    batch_col = adata.obs[HVG_BATCH_KEY]
    subsample_idx = []
    for batch in batch_col.unique():
        batch_idx = adata.obs_names[batch_col == batch]
        n_sample = min(HVG_SUBSAMPLE_PER_BATCH, len(batch_idx))
        subsample_idx.extend(np.random.choice(batch_idx, size=n_sample, replace=False))
    adata_hvg = adata[subsample_idx].copy()
    print(f"HVG 子采样：每 batch 取 {HVG_SUBSAMPLE_PER_BATCH} 细胞（实际 {len(subsample_idx)} 总细胞）用于 HVG 选择")
else:
    adata_hvg = adata

print(f"HVG Sweep: n_genes={_n_genes_values}, flavor={_flavor_values}")
print(f"BATCH_AWARE_HVG={BATCH_AWARE_HVG}", end="")
if BATCH_AWARE_HVG:
    print(f", batch_key={HVG_BATCH_KEY}")
else:
    print()
print()

for n_genes in _n_genes_values:
    for flavor in _flavor_values:
        label = f"n={n_genes}, flavor={flavor}"
        print(f"--- {label} ---")

        # batch-aware HVG：每个 batch 独立选 HVG，取并集按跨 batch 出现频率排名
        if BATCH_AWARE_HVG:
            if HVG_BATCH_KEY not in adata.obs.columns:
                raise KeyError(
                    f"HVG_BATCH_KEY='{HVG_BATCH_KEY}' 不在 obs 列中。"
                    f"可用列: {list(adata.obs.columns)[:10]}..."
                    f"请检查 BATCH_AWARE_HVG 和 HVG_BATCH_KEY 参数。"
                )
            sc.pp.highly_variable_genes(
                adata_hvg, n_top_genes=n_genes, flavor=flavor,
                batch_key=HVG_BATCH_KEY
            )
        else:
            sc.pp.highly_variable_genes(
                adata_hvg, n_top_genes=n_genes, flavor=flavor
            )

        # 子采样模式下将子集 HVG 投影回全数据
        if adata_hvg is not adata:
            adata.var["highly_variable"] = adata.var_names.isin(
                adata_hvg.var_names[adata_hvg.var["highly_variable"]]
            )

        # 存储结果到带参数后缀的 var 列，供后续对比
        key = f"hvg_{flavor}_{n_genes}"
        adata.var[key] = adata.var["highly_variable"].copy()
        n_selected = int(adata.var[key].sum())
        hvg_results[key] = {"n_genes": n_genes, "flavor": flavor, "n_selected": n_selected}
        print(f"  {key}: {n_selected} genes selected")

print(f"\n共生成 {len(hvg_results)} 组 HVG 结果")
print(f"adata.var['highly_variable'] 当前指向最后组合: {list(hvg_results.keys())[-1]}")

# 清理子采样临时对象（释放 adata 子集占用的内存）
if adata_hvg is not adata:
    del adata_hvg

# --- 子采样稳定性验证 ---
# 3 次不同 seed，检查 HVG 重叠度。如果 Jaccard < 0.8 → 建议增大 HVG_SUBSAMPLE_PER_BATCH
if HVG_SUBSAMPLE_PER_BATCH is not None:
    _stability_sets = []
    for _seed in [RANDOM_SEED, RANDOM_SEED + 1, RANDOM_SEED + 2]:
        np.random.seed(_seed)
        _sub_idx = []
        for batch in adata.obs[HVG_BATCH_KEY].unique():
            batch_idx = adata.obs_names[adata.obs[HVG_BATCH_KEY] == batch]
            n_s = min(HVG_SUBSAMPLE_PER_BATCH, len(batch_idx))
            _sub_idx.extend(np.random.choice(batch_idx, size=n_s, replace=False))
        _tmp = adata[_sub_idx].copy()
        sc.pp.highly_variable_genes(_tmp, n_top_genes=_n_genes_values[-1], flavor=_flavor_values[-1],
                                    batch_key=HVG_BATCH_KEY)
        _stability_sets.append(set(_tmp.var_names[_tmp.var["highly_variable"]]))
        del _tmp
    jaccards = []
    for s1, s2 in itertools.combinations(_stability_sets, 2):
        jaccards.append(len(s1 & s2) / len(s1 | s2))
    mean_j = np.mean(jaccards)
    print(f"HVG 子采样稳定性：3 次 Jaccard = {[f'{j:.3f}' for j in jaccards]} (mean={mean_j:.3f})")
    if mean_j < 0.8:
        print(f"  ⚠️ 子采样不稳定（Jaccard < 0.8），建议增大 HVG_SUBSAMPLE_PER_BATCH")
    else:
        print(f"  ✓ 子采样稳定（Jaccard ≥ 0.8）")

## HVG Sweep 对比

当使用列表模式（多个 `N_TOP_GENES` 或 `HVG_FLAVOR` 值）时，
下方 cell 输出不同参数组合间 HVG 集合的 Jaccard 相似度热力图。

**解读**：
- 对角线 = 1.0（自身完全一致）
- Jaccard 接近 1.0 的 pair → 参数选择对该对不敏感，可放心任选
- Jaccard 显著低于 1.0 的 pair → HVG 集合对参数敏感，建议 PI 目视下游 UMAP 效果后决定

单值模式下此 cell 仅打印确认信息，不画图。

In [ ]:
# === HVG Sweep 对比：Jaccard 重叠度热力图 ===
if len(hvg_results) > 1:
    # Jaccard 相似度矩阵：不同参数组合间 HVG 集合的重叠度
    keys = list(hvg_results.keys())
    jaccard_matrix = pd.DataFrame(index=keys, columns=keys, dtype=float)

    for k1, k2 in itertools.combinations(keys, 2):
        set1 = set(adata.var_names[adata.var[k1]])
        set2 = set(adata.var_names[adata.var[k2]])
        j = len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0.0
        jaccard_matrix.loc[k1, k2] = j
        jaccard_matrix.loc[k2, k1] = j
    np.fill_diagonal(jaccard_matrix.values, 1.0)

    # 热力图
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(jaccard_matrix.values.astype(float), cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(keys, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(keys)))
    ax.set_yticklabels(keys, fontsize=9)
    plt.colorbar(im, ax=ax, label="Jaccard similarity")
    ax.set_title("HVG 集合重叠度（不同参数组合间）", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("results/figures/03_hvg_jaccard_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

    # 汇总表
    print("\nHVG Sweep 汇总:")
    summary_df = pd.DataFrame(hvg_results).T
    from IPython.display import display as ipy_display
    ipy_display(summary_df)
else:
    entry = list(hvg_results.values())[0]
    print(f"✓ 单值模式：n_genes={entry['n_genes']}, flavor={entry['flavor']}, "
          f"选定 {entry['n_selected']} HVG")

## HVG 排除列表

某些基因类别表达量在不同细胞间确实高度可变，但这种可变性反映的是
**通用细胞状态**（如代谢活性、应激水平）或**技术背景**而非**细胞身份**——
它们不应驱动聚类和细胞类型鉴定。排除后依赖真正的细胞身份基因来区分细胞类型。

**排除类别**：
- **线粒体基因（MT-*）**：反映细胞膜破损/凋亡程度，是 QC 指标而非身份信号
- **核糖体基因（RPS*/RPL*）**：反映蛋白质合成活性，与细胞类型关联弱、批次间波动大
- **血红蛋白基因（HBA*/HBB*）**：红细胞污染标志，在组织样本中来自血液残留
- **自定义 pattern**：如免疫球蛋白（IG[HKL]）和 TCR（TR[ABGD]）基因——
  在淋巴细胞中高度可变，但反映的是克隆型而非细胞类型

**注意**：排除后**不做补位**。补位会把刚排除的基因类别中排名较低的成员重新拉回来，
削弱排除效果。如需精确某数量的 HVG，适当增大 `N_TOP_GENES` 初始值即可。

## 细胞周期评分（始终执行）

细胞周期评分（`sc.tl.score_genes_cell_cycle`）始终执行并将
`S_score`、`G2M_score`、`phase` 存入 `adata.obs`，无论
`EXCLUDE_CELL_CYCLE_FROM_HVG` 的值如何。

**为什么始终做评分？**
stage4（scVI 嵌入）可以将 `S_score`/`G2M_score` 作为 continuous
covariate 来消除周期效应，而无需从 HVG 中粗暴排除细胞周期基因。
这对研究增殖异常的项目（如肿瘤、炎-癌转化）尤为重要——
增殖信号本身就是重要的生物学信息。

**从 HVG 排除细胞周期基因**（`EXCLUDE_CELL_CYCLE_FROM_HVG=True`）
仅在增殖信号过度主导 PCA、掩盖细胞类型差异时作为最后手段启用。
默认保持 `False`。


In [ ]:
# === HVG 排除列表：构建排除 mask，移除后不补位 ===
exclude_mask = pd.Series(False, index=adata.var_names)
excluded_counts = {}

if EXCLUDE_MT_FROM_HVG:
    mt_mask = adata.var_names.str.upper().str.startswith("MT-")
    excluded_counts["MT"] = int((adata.var["highly_variable"] & mt_mask).sum())
    exclude_mask |= mt_mask

if EXCLUDE_RIBO_FROM_HVG:
    ribo_mask = adata.var_names.str.upper().str.match("^(RPS|RPL)")
    excluded_counts["Ribo"] = int((adata.var["highly_variable"] & ribo_mask).sum())
    exclude_mask |= ribo_mask

if EXCLUDE_HB_FROM_HVG:
    hb_mask = adata.var_names.str.upper().str.match("^HB[^P]")  # HBA/HBB 但不含 HBP
    excluded_counts["HB"] = int((adata.var["highly_variable"] & hb_mask).sum())
    exclude_mask |= hb_mask

for pattern in CUSTOM_EXCLUDE_PATTERNS:
    try:
        custom_mask = adata.var_names.str.contains(pattern, case=False, regex=True)
    except re.error as e:
        print(f"⚠ CUSTOM_EXCLUDE_PATTERNS 中 '{pattern}' 正则无效 ({e})，跳过")
        continue
    excluded_counts[pattern] = int((adata.var["highly_variable"] & custom_mask).sum())
    exclude_mask |= custom_mask

# === 细胞周期评分（始终执行，结果存 obs 供 stage4 作为 covariate 使用）===
# Tirosh et al. 2015 的 S 期和 G2M 期 marker genes
s_genes_cc = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
g2m_genes_cc = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]

# 评分需要 log-normalized 数据（standard 模式下当前 adata.X 已经是）
if NORMALIZATION_METHOD == "pearson_residuals":
    # score_genes_cell_cycle 需要 log-normalized 数据
    _tmp_cc = adata.copy()
    _tmp_cc.X = _tmp_cc.layers["counts"].copy()
    sc.pp.normalize_total(_tmp_cc, target_sum=1e4)
    sc.pp.log1p(_tmp_cc)
    sc.tl.score_genes_cell_cycle(_tmp_cc, s_genes=s_genes_cc, g2m_genes=g2m_genes_cc)
    adata.obs["S_score"] = _tmp_cc.obs["S_score"]
    adata.obs["G2M_score"] = _tmp_cc.obs["G2M_score"]
    adata.obs["phase"] = _tmp_cc.obs["phase"]
    del _tmp_cc
else:
    sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes_cc, g2m_genes=g2m_genes_cc)
print(f"细胞周期评分完成 → obs 新增: S_score, G2M_score, phase")
_phase_counts = adata.obs["phase"].value_counts()
for phase, count in _phase_counts.items():
    print(f"  {phase}: {count} ({100*count/adata.n_obs:.1f}%)")

# 可选：从 HVG 排除细胞周期基因（默认 False）
# 当增殖信号过度主导 PCA 时启用——优先用 stage4 covariate 消除周期效应
if EXCLUDE_CELL_CYCLE_FROM_HVG:
    cc_genes = set(s_genes_cc + g2m_genes_cc)
    cc_mask = adata.var_names.isin(cc_genes)
    excluded_counts["CellCycle"] = int((adata.var["highly_variable"] & cc_mask).sum())
    exclude_mask |= cc_mask

# 移除
n_before = int(adata.var["highly_variable"].sum())
adata.var.loc[exclude_mask, "highly_variable"] = False
n_after = int(adata.var["highly_variable"].sum())
n_removed = n_before - n_after

# 排除后数量不足警告
_requested = _n_genes_values[-1] if isinstance(N_TOP_GENES, list) else N_TOP_GENES
if n_after < _requested * 0.8:
    print(f"⚠️ HVG 排除后仅剩 {n_after}（< 80% of requested {_requested}），建议增大 N_TOP_GENES 或减少排除类别")

print(f"HVG 排除列表：移除 {n_removed} 基因")
for cat, count in excluded_counts.items():
    if count > 0:
        print(f"  {cat}: {count}")
print(f"剩余 HVG: {n_after}")
print(f"（不做补位——如需精确数量请增大 N_TOP_GENES 初始值）")

## 强制纳入关键基因

`FORCED_INCLUDE_GENES` 允许 PI 将特定关键基因强制加入 HVG 列表，
即使这些基因在统计上未达到高可变阈值。

**为什么需要强制纳入？**
研究者关心的关键生物学 axis（如炎-癌转化 marker：
`CDX2`、`TFF3`、`GKN1`、`MUC2`、`OLFM4`、`LGR5` 等）
可能在当前数据集中的表达变异度未排进前 N——
但这不代表它们不重要。强制纳入保证它们一定进入下游 PCA 空间，
从而在聚类、UMAP 可视化和细胞类型鉴定中被利用。

加 20-30 个关键基因不会实质性改变 HVG 整体空间结构，
但能确保研究者关注的生物学信号不被遗漏。


In [ ]:
# === 强制纳入关键基因 ===
# 在 HVG 排除逻辑之后执行——PI 指定的关键基因强制设为 highly_variable=True
if FORCED_INCLUDE_GENES:
    found = [g for g in FORCED_INCLUDE_GENES if g in adata.var_names]
    not_found = [g for g in FORCED_INCLUDE_GENES if g not in adata.var_names]
    n_before = int(adata.var["highly_variable"].sum())
    adata.var.loc[found, "highly_variable"] = True
    n_after = int(adata.var["highly_variable"].sum())
    newly_added = n_after - n_before
    print(f"强制纳入: {len(found)} 基因找到, {newly_added} 新增到 HVG (总 HVG: {n_after})")
    if not_found:
        print(f"  ⚠️ 未找到（检查基因名大小写）: {not_found}")
else:
    print("FORCED_INCLUDE_GENES 为空，跳过强制纳入")


In [ ]:
# HVG 组成分析：选中的 HVG 是什么类别的基因？
# 如果某类别占比异常高 → 需要调整排除列表或 N_TOP_GENES
hvg_genes = adata.var_names[adata.var["highly_variable"]]
n_hvg = len(hvg_genes)

categories = {
    "MT（线粒体）": hvg_genes.str.upper().str.startswith("MT-").sum(),
    "Ribo（核糖体）": hvg_genes.str.upper().str.match("^(RPS|RPL)").sum(),
    "HB（血红蛋白）": hvg_genes.str.upper().str.match("^HB[^P]").sum(),
    "IG（免疫球蛋白）": hvg_genes.str.upper().str.match("^IG[HKL]").sum(),
}

# Cell Cycle（Tirosh 2015 S + G2M 期 marker genes）
# 若 EXCLUDE_CELL_CYCLE_FROM_HVG=False，仅统计不排除，供 PI 判断是否需要排除
_s = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
_g2m = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
categories["Cell Cycle（细胞周期）"] = hvg_genes.isin(set(_s + _g2m)).sum()

categories["其他"] = n_hvg - sum(categories.values())

print(f"===== HVG 组成分析（{n_hvg} genes）=====")
for cat, count in categories.items():
    pct = 100 * count / n_hvg if n_hvg > 0 else 0
    flag = " ⚠️" if pct > 5 and cat != "其他" else ""
    print(f"  {cat}: {count} ({pct:.1f}%){flag}")

# 如果 IG 基因占比 > 5%，说明 B 细胞过于突出
if categories.get("IG（免疫球蛋白）", 0) / max(n_hvg, 1) > 0.05:
    print("\n⚠️ IG 基因占比 > 5%：B 细胞信号可能主导聚类")
    print("  → 考虑将 IG 基因加入 CUSTOM_EXCLUDE_PATTERNS: [\"^IG[HKL]\"]")

In [ ]:
# === Marker 库覆盖率诊断 ===
# 检查 gastric_TEST_markers.csv 中的 marker 基因有多少被当前 HVG 选中
# 未被选中的 marker 不会参与下游聚类——影响对应细胞类型的鉴定
_marker_path = os.path.join("references", "markers", "gastric_TEST_markers.csv")
if os.path.exists(_marker_path):
    _marker_df = pd.read_csv(_marker_path, comment="#")
    # 兼容列名：尝试 gene / gene_symbol / Gene / marker
    _gene_col = None
    for col in ["gene", "gene_symbol", "Gene", "marker"]:
        if col in _marker_df.columns:
            _gene_col = col
            break
    if _gene_col:
        _marker_genes = _marker_df[_gene_col].dropna().unique().tolist()
        _in_var = [g for g in _marker_genes if g in adata.var_names]
        _in_hvg = [g for g in _in_var if adata.var.loc[g, "highly_variable"]]
        _not_in_hvg = [g for g in _in_var if not adata.var.loc[g, "highly_variable"]]
        print(f"\n===== Marker 库覆盖率（{_marker_path}）=====")
        print(f"  Marker 总基因: {len(_marker_genes)}")
        print(f"  在 adata 中: {len(_in_var)}")
        print(f"  在 HVG 中: {len(_in_hvg)} ({100*len(_in_hvg)/max(len(_in_var),1):.1f}%)")
        if _not_in_hvg:
            print(f"  未被选入 HVG: {_not_in_hvg[:20]}")  # 最多显示 20 个
            print(f"  → 如需这些基因参与聚类，加入 FORCED_INCLUDE_GENES")
    else:
        print(f"Marker CSV 列名不匹配，期望 gene/gene_symbol/Gene/marker 之一")
else:
    print(f"未找到 marker 文件 {_marker_path}，跳过覆盖率检查")


In [ ]:
# === 胃粘膜关键谱系基因 HVG 覆盖诊断 ===
# GCPL 炎-癌转化研究中，这些谱系特异性基因必须参与聚类——
# 如果某 lineage 的核心 marker 未被 HVG 选中，下游 UMAP/PCA 将无法分辨该细胞群体
_GASTRIC_LINEAGE_PANEL = {
    "壁细胞": ["ATP4A", "ATP4B", "GIF"],
    "主细胞": ["PGA3", "PGA4", "PGA5", "LIPF", "PGC"],
    "表面黏液": ["MUC5AC", "TFF1", "GKN1", "GKN2"],
    "颈黏液": ["MUC6", "TFF2"],
    "SPEM": ["WFDC2", "CD44", "AQP5"],
    "肠化": ["CDX2", "MUC2", "TFF3", "VIL1", "OLFM4"],
    "内分泌": ["CHGA", "CHGB", "SYP"],
}

print("\n===== 胃粘膜关键谱系基因 HVG 覆盖 =====\n")
_total_panel = 0
_total_in_hvg = 0
_missing_critical = []

for lineage, genes in _GASTRIC_LINEAGE_PANEL.items():
    _in_var = [g for g in genes if g in adata.var_names]
    _in_hvg = [g for g in _in_var if adata.var.loc[g, "highly_variable"]]
    _not_hvg = [g for g in _in_var if not adata.var.loc[g, "highly_variable"]]
    _total_panel += len(_in_var)
    _total_in_hvg += len(_in_hvg)

    _status = "✓" if len(_not_hvg) == 0 else "⚠️"
    print(f"  {_status} {lineage}: {len(_in_hvg)}/{len(_in_var)} 在 HVG 中", end="")
    if _not_hvg:
        print(f"  (未覆盖: {_not_hvg})")
        _missing_critical.extend(_not_hvg)
    else:
        print()

_coverage_pct = 100 * _total_in_hvg / max(_total_panel, 1)
print(f"\n  总覆盖率: {_total_in_hvg}/{_total_panel} ({_coverage_pct:.0f}%)")

if _missing_critical:
    print(f"\n  → {len(_missing_critical)} 个关键谱系基因未进入 HVG:")
    print(f"    {_missing_critical}")
    print(f"    建议加入 FORCED_INCLUDE_GENES 参数确保聚类可见")
else:
    print(f"\n  ✓ 所有关键谱系基因均在 HVG 中——GCPL 分析就绪")

## HVG 诊断图

**平均表达量 vs 离散度图**：每个点是一个基因。X 轴是平均表达量（log scale），
Y 轴是标准化离散度。蓝色点为选中的 HVG，灰色点为未选中。
理想情况下蓝色点应在各个表达水平上均匀覆盖高离散度区域。

**跨批次出现频率图**（batch-aware 模式专属）：展示 HVG 在多少个 batch 中被选中。
出现频率越高的基因越可能是跨数据集保守的真信号，而非单数据集的技术噪声。

In [ ]:
# === HVG 诊断图 ===
# 图1：平均表达量 vs 标准化离散度（标准 scanpy 诊断图）
sc.pl.highly_variable_genes(adata, show=True)
plt.savefig("results/figures/03_hvg_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()
print("HVG 诊断图已保存至 results/figures/03_hvg_diagnostic.png")

# 图2：batch-aware 时额外输出——per-batch HVG 出现频率分布
if BATCH_AWARE_HVG and "highly_variable_nbatches" in adata.var.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    nbatches_dist = adata.var["highly_variable_nbatches"].value_counts().sort_index()
    nbatches_dist.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_xlabel("出现在几个 batch 中")
    ax.set_ylabel("基因数")
    ax.set_title("HVG 跨 batch 出现频率\n（越多 batch 共享 = 越可能是真信号）")
    plt.tight_layout()
    plt.savefig("results/figures/03_hvg_nbatches.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    # 统计摘要
    n_total_batches = adata.obs[HVG_BATCH_KEY].nunique()
    n_ubiquitous = int((adata.var["highly_variable_nbatches"] == n_total_batches).sum())
    print(f"总 batch 数: {n_total_batches}")
    print(f"在所有 {n_total_batches} 个 batch 中均为 HVG 的基因: {n_ubiquitous}")
elif BATCH_AWARE_HVG:
    print("（batch-aware 模式下未检测到 highly_variable_nbatches 列，可能 flavor 不支持此输出）")

## （可选）回归混杂变量

`sc.pp.regress_out` 用线性回归从表达矩阵中移除指定协变量的影响。

**为什么不推荐在此阶段回归？**
1. 回归会 **densify** 稀疏矩阵，内存占用可能膨胀 10-50 倍
2. 下游 Harmony/scVI 已内置批次/协变量处理，通常无需提前回归
3. 线性回归假设协变量与表达量是线性关系——对单细胞 counts 数据往往不成立

**什么时候启用？** PI 在 04 嵌入后发现某个混杂因素（如 MT% 或 total_counts）
严重主导了 PCA/UMAP 且 Harmony/scVI 无法充分校正时，才考虑回归。

In [ ]:
# === 回归混杂变量（可选）===
if REGRESS_OUT:
    print(f"⚠ 回归混杂变量：{REGRESS_OUT}")
    print("  注意：回归会 densify 矩阵，大幅增加内存。通常不推荐——")
    print("  更好的做法是在 04 嵌入中通过 batch_key/covariates 处理。")

    # 检查所有回归变量是否存在于 adata.obs
    missing = [v for v in REGRESS_OUT if v not in adata.obs.columns]
    if missing:
        raise KeyError(f"REGRESS_OUT 中包含不存在于 adata.obs 的列: {missing}。"
                       f"可用列: {list(adata.obs.columns)}")

    # NaN 值防御：某些变量可能因上游跳过而含 NaN
    _valid_regress = []
    for var in REGRESS_OUT:
        if var not in adata.obs.columns:
            print(f"  ⚠️ 跳过 '{var}'：不在 obs 中")
            continue
        nan_pct = adata.obs[var].isna().mean()
        if nan_pct > 0.05:
            print(f"  ⚠️ 跳过 '{var}'：{nan_pct:.1%} NaN（> 5% 阈值）")
            continue
        elif nan_pct > 0:
            # 少量 NaN：填充中位数
            adata.obs[var] = adata.obs[var].fillna(adata.obs[var].median())
            print(f"  '{var}': {nan_pct:.1%} NaN 已填充为中位数")
        _valid_regress.append(var)

    if _valid_regress:
        # 回归前的内存快照
        was_sparse = sp.issparse(adata.X)
        sc.pp.regress_out(adata, _valid_regress)
        adata.X = adata.X.astype(np.float32)

        # 尝试恢复 sparse（如果足够稀疏）
        if not sp.issparse(adata.X):
            zero_fraction = (adata.X == 0).mean()
            if zero_fraction > 0.5:
                adata.X = sp.csr_matrix(adata.X)
                print(f"  回归后恢复 sparse（稀疏度 {zero_fraction:.1%}）")
            else:
                print(f"  ⚠ 回归后矩阵不够稀疏（零值占比 {zero_fraction:.1%}），保持 dense")
                print(f"    内存占用约 {adata.X.nbytes / 1024**2:.0f} MB")
    else:
        print("  所有回归变量均不可用或含过多 NaN，跳过回归")
else:
    print("✓ 跳过回归（REGRESS_OUT 为空）")

## （可选）缩放

`sc.pp.scale` 将每个基因的表达量标准化到均值为 0、方差为 1——
使 PCA 中各基因权重均等，避免高表达基因主导主成分。

**注意**：
- scale 会 **densify** 矩阵
- **Harmony/scVI 不需要 scale**——Harmony 在 PCA 空间校正、scVI 用原始 counts + 内部归一化
- PCA-only 管线（不做任何批次校正的简单分析）可能需要
- scale 前会保存 normalized 层到 `adata.layers["normalized"]`，以便后续可恢复

In [ ]:
# === 缩放（可选）===
if SCALE:
    print(f"缩放：max_value={MAX_SCALE_VALUE}")
    print("  注意：scale 会 densify 矩阵。Harmony/scVI 不需要 scale。")

    # 保存 normalized 到 layer 以便后续可恢复 sparse
    adata.layers["normalized"] = adata.X.copy()
    sc.pp.scale(adata, max_value=MAX_SCALE_VALUE)
    adata.X = adata.X.astype(np.float32)
    print(f"  缩放后 X mean={adata.X.mean():.4f}, std={adata.X.std():.4f}")
else:
    print("✓ 跳过缩放（SCALE=False，Harmony/scVI 不需要）")

## Sanity PCA 诊断（轻量级，不修改 adata）

在写入 checkpoint 前执行一次轻量级 PCA（仅 5 个主成分），
帮助 PI 快速判断标准化是否干净、batch 效应有多强。

**诊断项**：
- **PC1 vs library size**：标准化后 PC1 仍与文库大小相关 → 标准化不够彻底
- **PC1 ANOVA by batch**：PC1 在各 batch 间差异显著 → batch effect 需要重点处理
- **方差解释比**：PC1 占比 > 30% → 可能存在强混杂因素

**注意**：此 PCA 使用临时 `adata` 子集，不修改 `adata.obsm`/`adata.uns`，
不影响后续 stage4 的正式 PCA。


In [ ]:
# === Sanity PCA 诊断（轻量级，不修改 adata）===
print("\n===== 标准化效果快速诊断 =====")
_hvg_mask = adata.var["highly_variable"]
_tmp_pca = adata[:, _hvg_mask].copy()

# scale + PCA（临时对象，不影响 adata）
sc.pp.scale(_tmp_pca, max_value=10)
sc.tl.pca(_tmp_pca, n_comps=5, random_state=RANDOM_SEED)

# PC1 vs library size
_log_lib = np.log1p(np.array(adata.layers["counts"].sum(axis=1)).flatten())
_pc1 = _tmp_pca.obsm["X_pca"][:, 0]
_r_lib = np.corrcoef(_log_lib, _pc1)[0, 1]
print(f"  PC1 vs log(total_counts): r = {_r_lib:.3f}", end="")
if abs(_r_lib) > 0.3:
    print(" ⚠️ 标准化后 PC1 仍受 library size 驱动")
else:
    print(" ✓ library size 效应已消除")

# PC1 vs batch (ANOVA)
if HVG_BATCH_KEY in adata.obs.columns and adata.obs[HVG_BATCH_KEY].nunique() > 1:
    from scipy.stats import f_oneway
    _groups = [_pc1[adata.obs[HVG_BATCH_KEY] == b] for b in adata.obs[HVG_BATCH_KEY].unique()]
    _f_stat, _p_val = f_oneway(*_groups)
    print(f"  PC1 ANOVA by {HVG_BATCH_KEY}: F={_f_stat:.1f}, p={_p_val:.2e}", end="")
    if _p_val < 1e-10:
        print(" → batch effect 显著，stage4 整合方法需重点处理")
    else:
        print(" ✓ batch 不主导 PC1")

# 方差解释比
_var_ratio = _tmp_pca.uns["pca"]["variance_ratio"]
print(f"  PCA 方差解释: PC1={_var_ratio[0]:.3f}, PC2={_var_ratio[1]:.3f}, PC3={_var_ratio[2]:.3f}")
if _var_ratio[0] > 0.3:
    print("  ⚠️ PC1 方差占比 > 30%——可能存在强混杂因素，检查 batch/library_size/MT%")

del _tmp_pca
gc.collect()


## 参数记录 + Checkpoint

将标准化参数写入 `adata.uns` 以便追溯，执行内存自检确保数据完整性，
然后将产出写入磁盘形成 03 checkpoint。

**写入的 uns 字段**：
- `normalize_v1` — 本次运行的完整参数记录
- `stage` / `status` / `upstream` / `version` — 管线追溯链

**内存自检**：确认 `adata.X` 是 float32（sparse 或 dense 均可，
因 regress_out 或 scale 可能导致 dense——允许但警告）。

### Stage 03 Verdict

本 stage 完成后应确认：
- [ ] PC1 vs library size 相关 < 0.3（标准化有效）
- [ ] HVG 覆盖关键胃粘膜谱系基因（见上方诊断）
- [ ] 细胞周期评分已存入 obs（供 04 使用）


In [ ]:
# === 参数记录 + Checkpoint ===

# 记录运行参数
adata.uns["normalize_v1"] = {
    "method": NORMALIZATION_METHOD,
    "target_sum": TARGET_SUM if NORMALIZATION_METHOD == "standard" else None,
    "n_top_genes": N_TOP_GENES if not isinstance(N_TOP_GENES, list) else _n_genes_values[-1],
    "hvg_flavor": HVG_FLAVOR if not isinstance(HVG_FLAVOR, list) else _flavor_values[-1],
    "batch_aware": BATCH_AWARE_HVG,
    "hvg_batch_key": HVG_BATCH_KEY if BATCH_AWARE_HVG else None,
    "excluded_from_hvg": excluded_counts,
    "forced_include_genes": FORCED_INCLUDE_GENES,
    "regress_out": REGRESS_OUT,
    "scale": SCALE,
}
print("normalize_v1:", adata.uns["normalize_v1"])

# 版本追踪——供迭代回跑追溯链使用
adata.uns["stage"] = "03_normalized"
adata.uns["status"] = "experimental"          # PI 审查后手动改为 "promoted"
adata.uns["upstream"] = [UPSTREAM_PATH]       # 上游文件，完整溯源链
adata.uns["version"] = f"v{OUTPUT_VERSION}"   # 与 OUTPUT_PATH 版本号一致

# --- 写入前检查 ---
if sp.issparse(adata.X):
    assert adata.X.dtype == np.float32, (
        f"dtype 应为 float32，实际 {adata.X.dtype}"
    )
    print("内存自检通过: X 是 sparse float32")
else:
    # regress_out 或 scale 可能导致 dense，允许但警告
    print(f"⚠ 矩阵为 dense（可能因 regress_out 或 scale），写入会较大")
    print(f"  内存占用约 {adata.X.nbytes / 1024**2:.0f} MB")
    assert adata.X.dtype == np.float32, (
        f"dtype 应为 float32，实际 {adata.X.dtype}"
    )

# --- 写入 checkpoint ---
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"✓ 写入 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes, "
      f"HVG={int(adata.var['highly_variable'].sum())})")

# 校验文件正确写出
assert os.path.exists(OUTPUT_PATH), f"Output NOT found: {OUTPUT_PATH}"
print(f"Verified: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

# --- 释放内存 ---
# 跨 stage 边界释放内存，避免在同一 kernel 中累积
del adata
gc.collect()
print("Memory released.")